#### Class -- Magic Method



魔法方法`（Magic Methods）`，也叫 `Dunder Methods`（`Double Underscore` 的缩写），是 Python 中以双下划线开头和结尾的特殊方法（例如 `__init__`、`__call__`）。

核心特点：
- 官方内置：必须使用 Python 官方规定的名字。无法自己定义（比如写 `__my_func__` 是无效的），
- 隐式触发（语法糖）：几乎不会直接用 `obj.__call__()` 调用，而是使用 `Python` 的特定语法（如 `obj()`、`len(obj)`、`obj[0]`、`with obj:`），`Python` 解释器会在底层自动去调用对应的魔法方法。
- 定制类的行为：它们赋予了自定义对象和 `Python` 内置对象（如列表、字典、函数）一样自然的操作方式。

---
##### 生命周期与初始化：`__init__ 与 super()`

In [ ]:
class StandaloneClass:
    def __init__(self, name):

        # 执行父类准备好的内部逻辑。
        # 写 super().__init__() 永远是最安全的选择
        super().__init__() 
        self.name = name
        print(f"{self.name}  initialized")

obj = StandaloneClass("test")
print(obj.name) #  test

 test 已初始化
test


##### 容器与序列行为：`__len__ 与 __getitem__`


In [ ]:
class Playlist:
    def __init__(self, name, songs):
        self.name = name
        self.songs = songs
        
    # 触发方式：len(obj)
    def __len__(self):
        return len(self.songs)
    
    # 触发方式：obj[idx]
    def __getitem__(self, idx):
        return self.songs[idx]

my_playlist = Playlist("日常歌单", ["歌曲A", "歌曲B", "歌曲C"])
print(len(my_playlist))    #  3 
print(my_playlist[1])      #  "歌曲B" 

# 即使没有定义 __iter__，Python 也会利用 __getitem__ 进行迭代遍历
for song in my_playlist:
    print(song)

`__getitem__`必须接收 idx 参数，返回的数据类型应保持统一，例如都返回 Tensor 和整数类型标签。

##### 可调用对象 `__call__`

- 触发方式是 `obj(...)`

- 作用在于让实例对象变得像普通函数一样可以被直接调用

这在需要封装状态但又希望像函数一样使用的场景中非常有用

---
`PyTorch` 允许写 `model(inputs)` 而不是 `model.forward(inputs)`，因为在 `nn.Module` 父类中重写了 `__call__` 方法。

当写 model(inputs) 时 Python 自动触发了 `model.__call__(inputs)`，父类在 `__call__`内部做了一堆准备工作 比如处理前置钩子 hook 控制梯度，然后再去调用手写的 `forward()`，这是禁止直接写 `model.forward(x)` 的原因

In [5]:
class Multiplier:
    def __init__(self, factor):
        self.factor = factor

    def __call__(self, number):
        return number * self.factor

double = Multiplier(2)
print(double(5)) #  10

10


##### 字符串表示 `__str__` 与 `__repr__`

- 触发方式 `print(obj)` 或直接在终端敲 `obj` 回车。

- 定义对象的打印外貌，让打印出来的信息具备可读性，而不是默认的内存地址。
- `__str__` 面向用户，注重可读性，比如打印友好的提示信息。
- `__repr__` 面向开发者，注重准确性，通常返回一个能够重新创建该对象的字符串表达式。
- 当 `__str__` 没有定义时，Python 会退而求其次调用 `__repr__`。

---
- 当直接打印 `print(model)` 时，PyTorch 的 `nn.Module` 重写了这两个方法，自动生成一张极其漂亮的网络结构表，显示了每一层的名称、输入输出维度。

In [6]:
class Book:
    def __init__(self, title, author):
        self.title = title
        self.author = author

    def __str__(self):
        return f"书籍：{self.title}，作者：{self.author}"

    def __repr__(self):
        return f"Book('{self.title}', '{self.author}')"

my_book = Book("深度学习", "李四")
print(my_book)          # 触发 __str__
print(repr(my_book))    # 触发 __repr__

书籍：深度学习，作者：李四
Book('深度学习', '李四')


##### 算术运算符重载 `__add__` 与 `__mul__`

- 触发方式
- `obj1 + obj2` 触发 `__add__`
- `obj1 * obj2` 触发 `__mul__`

- 作用
- 让自定义对象能够像数字一样参与算术运算。
- 当执行 `a + b` 时，Python 底层实际上是尝试调用 `a.__add__(b)`。如果 `a` 没有实现 `__add__`，或者返回了 `NotImplemented`，Python 会尝试调用 `b.__radd__(a)`（反向加法）。
- 除了 `__add__`，还有 `__sub__`（减法）、`__mul__`（乘法）、`__truediv__`（除法）等一整套算术魔法方法。

---
- `torch.Tensor` 类重写了大量的算术魔法方法。因此才能自然地写出 `tensor_a + tensor_b` 或者 `tensor * 2`。
- 实际上相当于调用了 `tensor_a.__add__(tensor_b)`，底层执行了逐元素的加法操作，并返回一个全新的张量。

In [ ]:
class Vector:
    def __init__(self, x, y):
        self.x = x
        self.y = y

    def __add__(self, other):
        return Vector(self.x + other.x, self.y + other.y)

    def __mul__(self, scalar):
        return Vector(self.x * scalar, self.y * scalar)

    def __repr__(self):
        return f"Vector({self.x}, {self.y})"

v1 = Vector(1, 2)
v2 = Vector(3, 4)

print(v1 + v2)    # 触发 __add__，输出 Vector(4, 6)
print(v1 * 3)     # 触发 __mul__，输出 Vector(3, 6)

##### 比较魔法方法 `__eq__` `__lt__` 与 `__gt__`

- 触发方式
- `obj1 == obj2` 触发 `__eq__`
- `obj1 < obj2` 触发 `__lt__`
- `obj1 > obj2` 触发 `__gt__`

- 作用
- 让自定义对象支持比较操作。
- 默认情况下，自定义对象只能比较内存地址（`is`）。重写这些方法后，可以基于对象的属性进行逻辑比较。
- 如果比较的操作数类型不支持，应当返回 `NotImplemented`，让 Python 尝试调用反向比较方法（如 `__gt__` 对应 `__lt__`）。

---
- `torch.Tensor` 重写了这些方法。当你写 `tensor > 0` 时，底层触发的是 `tensor.__gt__(0)`，返回的是一个形状相同的布尔张量，而不是单个 True 或 False。
- 注意：张量的比较是逐元素的，这与 Python 内置类型的比较规则不同。

In [ ]:
class Version:
    def __init__(self, major, minor):
        self.major = major
        self.minor = minor

    def __eq__(self, other):
        return (self.major, self.minor) == (other.major, other.minor)

    def __lt__(self, other):
        return (self.major, self.minor) < (other.major, other.minor)

v1 = Version(1, 2)
v2 = Version(1, 3)
print(v1 == v2)  # 触发 __eq__，输出 False
print(v1 < v2)   # 触发 __lt__，输出 True

##### 属性访问魔法方法 `__getattr__` 与 `__setattr__`

- 触发方式
- 访问不存在的属性 `obj.xxx` 时触发 `__getattr__`
- 设置属性 `obj.xxx = value` 时触发 `__setattr__`

- 作用
- `__getattr__` 仅在常规属性查找失败时被调用，常用于动态代理或返回默认值。
- `__setattr__` 拦截所有的属性赋值操作，可以用来做合法性检查或日志记录。
- 这两个方法非常底层，滥用会导致代码难以调试。

---
- `nn.Module` 内部重写了 `__setattr__`。当你写 `self.conv1 = nn.Conv2d(...)` 时，它不仅仅是给对象赋了一个属性，还会自动将这个层注册到 `self._modules` 字典中。
- 正是因为 `__setattr__` 的拦截，PyTorch 才能自动追踪所有的层和参数，从而使得 `model.parameters()` 能够收集到它们并交给优化器。
- `__getattr__` 被用来动态处理一些不存在的属性，比如在分布式训练中查找特定的钩子。

In [ ]:
class Container:
    def __init__(self):
        self.data = {}

    def __setattr__(self, key, value):
        if key == "data":
            super().__setattr__(key, value)
        else:
            self.data[key] = value

    def __getattr__(self, key):
        if key in self.data:
            return self.data[key]
        raise AttributeError(f"没有属性 {key}")

c = Container()
c.color = "red"     # 触发 __setattr__，存入 data
print(c.color)      # 触发 __getattr__，从 data 读取

##### 布尔值魔法方法 `__bool__`

- 触发方式
- `if obj:` 或 `bool(obj)`

- 作用
- 定义对象的真假值。
- 如果不重写，对象默认总是为 True。
- 如果重写了 `__len__`，而没有重写 `__bool__`，Python 会调用 `__len__`，若长度为 0 则返回 False。

---
- `torch.Tensor` 重写了 `__bool__`，但是有一个严格限制：如果张量中的元素多于一个，使用 `if tensor:` 会抛出异常 `RuntimeError: Boolean value of Tensor with more than one value is ambiguous`。
- 如果确实需要判断张量真假，必须使用 `if tensor.all():` 或 `if tensor.any():`，或先用 `.item()` 提取单个元素。

class CustomBool:
    def __init__(self, value):
        self.value = value

    def __bool__(self):
        return self.value > 0

c1 = CustomBool(5)
c2 = CustomBool(-1)
print(bool(c1))  # 输出 True
print(bool(c2))  # 输出 False

##### 原地运算魔法方法 `__iadd__`

- 触发方式
- `obj += other`

- 作用
- 定义 += 操作的行为。
- 如果类没有实现 `__iadd__`，Python 会退而求其次使用 `__add__`，此时 `obj += other` 等价于 `obj = obj + other`，会创建一个新对象。
- 实现 `__iadd__` 可以实现原地修改，节省内存。

---
- `tensor += 1` 与 `tensor = tensor + 1` 在 PyTorch 中有本质区别。
- `tensor += 1` 会触发 `__iadd__`，进行原地操作，不创建新对象，内存地址不变。
- `tensor = tensor + 1` 会创建全新的张量对象。
- 在训练循环中，部分参数更新操作如果使用原地操作，可以节省显存，但有时会导致自动求导机制报错，因此需要谨慎使用。

In [ ]:
class Counter:
    def __init__(self, count):
        self.count = count

    def __iadd__(self, other):
        self.count += other
        return self

    def __repr__(self):
        return f"Counter({self.count})"

c = Counter(1)
original_id = id(c)
c += 5           # 触发 __iadd__，原地修改
print(c)         # 输出 Counter(6)
print(id(c) == original_id)  # 输出 True，说明没有创建新对象